# Communities and Crime — evaluacija i poređenje

Ova sveska:
1. Učitava sve rezultate klasterovanja
2. Pravi sveobuhvatnu tabelu poređenja metrika
3. Vizuelizuje poređenje metrika po algoritmima i skupovima karakteristika
4. Interpretira klastere — kakve zajednice su završile u kojoj grupi?

> **Napomena:** prvo pokrenuti sveske **`01`** i **`02`**.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded.')

Libraries loaded.


## 1. Učitavanje podataka i rezultata klasterovanja

In [4]:
X_full    = pd.read_csv('data_preprocessed.csv')
X_imputed = pd.read_csv('data_imputed.csv')
X_pca70   = pd.read_csv('data_pca_70.csv')
X_pca80   = pd.read_csv('data_pca_80.csv')
X_econ    = pd.read_csv('data_economic.csv')
targets   = pd.read_csv('data_targets.csv')

kmeans_results  = joblib.load('results_kmeans.pkl')
agg_results     = joblib.load('results_agglomerative.pkl')
dbscan_results  = joblib.load('results_dbscan.pkl')
clarans_results = joblib.load('results_clarans.pkl')
birch_results   = joblib.load('results_birch.pkl')
cure_results    = joblib.load('results_cure.pkl')

pca_viz = PCA(n_components=2, random_state=42)
X_viz = pca_viz.fit_transform(X_full)

results_df = pd.read_csv('clustering_results.csv')
print(f'Loaded {len(results_df)} algorithm-feature configurations.')
results_df.head()

Loaded 21 algorithm-feature configurations.


,Algorithm,Feature Set,n_clusters,Silhouette,Davies-Bouldin,Calinski-Harabasz
0,K-Means,Full,3,0.1777,1.7846,370.6
1,K-Means,PCA-70,3,0.2686,1.3279,606.3
2,K-Means,PCA-80,3,0.2277,1.5045,502.4
3,K-Means,Economic,3,0.2065,1.6030,720.4
4,Agglomerative (Ward),Full,3,0.1604,1.8891,324.6


## 2. Tabelarni prikaz poređenja metrika

- **Silueta koeficijent** [-1, 1]: što je veći, to je bolje. Meri koliko je tačka slična svom klasteru u poređenju sa ostalima.
- **Davies-Bouldinov indeks** [0, ∞): što je manji, to je bolje. Predstavlja odnos raspršenosti unutar klastera i razdvojenosti između klastera.
- **Calinski-Harabasz indeks** [0, ∞): što je veći, to je bolje. Predstavlja odnos disperzije između klastera i disperzije unutar klastera.

In [5]:
print('=== All Configurations — Sorted by Silhouette Score ===')
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
results_df.sort_values('Silhouette', ascending=False)

=== All Configurations — Sorted by Silhouette Score ===


,Algorithm,Feature Set,n_clusters,Silhouette,Davies-Bouldin,Calinski-Harabasz
19,CURE,PCA-80,3,0.7504,0.2869,93.0000
20,CURE,Economic,3,0.6796,0.3130,47.9000
18,CURE,PCA-70,3,0.6561,0.4540,110.1000
8,DBSCAN,PCA-70,3,0.5612,0.6230,62.3000
1,K-Means,PCA-70,3,0.2686,1.3279,606.3000
15,BIRCH,PCA-70,3,0.2429,1.4122,540.3000
5,Agglomerative (Ward),PCA-70,3,0.2325,1.4068,504.6000
2,K-Means,PCA-80,3,0.2277,1.5045,502.4000
13,CLARANS,Economic,3,0.2153,1.6814,611.0000
7,Agglomerative (Ward),Economic,3,0.2118,1.6467,608.0000


In [11]:
print('=== Top 5 by Silhouette ===')
top5_sil = results_df.dropna(subset=['Silhouette']).sort_values('Silhouette', ascending=False).head(5)
print(top5_sil[['Algorithm', 'Feature Set', 'n_clusters', 'Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']].to_string(index=False))

print('\n=== Top 5 by Davies-Bouldin ===')
top5_db = results_df.dropna(subset=['Davies-Bouldin']).sort_values('Davies-Bouldin').head(5)
print(top5_db[['Algorithm', 'Feature Set', 'n_clusters', 'Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']].to_string(index=False))

print('\n=== Top 5 by Calinski-Harabasz ===')
top5_ch = results_df.dropna(subset=['Calinski-Harabasz']).sort_values('Calinski-Harabasz', ascending=False).head(5)
print(top5_ch[['Algorithm', 'Feature Set', 'n_clusters', 'Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']].to_string(index=False))

=== Top 5 by Silhouette ===
Algorithm Feature Set  n_clusters  Silhouette  Davies-Bouldin  Calinski-Harabasz
     CURE      PCA-80           3      0.7504          0.2869            93.0000
     CURE    Economic           3      0.6796          0.3130            47.9000
     CURE      PCA-70           3      0.6561          0.4540           110.1000
   DBSCAN      PCA-70           3      0.5612          0.6230            62.3000
  K-Means      PCA-70           3      0.2686          1.3279           606.3000

=== Top 5 by Davies-Bouldin ===
Algorithm Feature Set  n_clusters  Silhouette  Davies-Bouldin  Calinski-Harabasz
     CURE      PCA-80           3      0.7504          0.2869            93.0000
     CURE    Economic           3      0.6796          0.3130            47.9000
     CURE      PCA-70           3      0.6561          0.4540           110.1000
   DBSCAN      PCA-70           3      0.5612          0.6230            62.3000
   DBSCAN      PCA-80           7      0.0393   